In [0]:
import pyspark.sql.functions as F
import time
import statistics

In [0]:
def _benchmark(df):
    """ Returns runtime of example spark expression (result)"""

    start = time.perf_counter()

    result = (
        df
        .agg(
            F.count("*").alias("row_count"),
            F.countDistinct("country").alias("distinct_countries")
        )
        .collect()
    )

    end = time.perf_counter()
    
    return end - start

In [0]:
def _benchmark_multiple(df, runs: int = 10):
    """Executes _benchmark() specified times"""

    times = []

    for i in range(runs):
        times.append(_benchmark(df))
    
    return statistics.median(times)


In [0]:
def _benchmark_skew(df, column):
    """Returns runtime of skewed DF"""

    start = time.perf_counter()

    (df
     .groupBy(column)
     .count()
     .collect()
     )

    end = time.perf_counter()

    return end - start

In [0]:
def _benchmark_skew_multiple(df, column, runs: int = 10):
    """Executes _benchmark() specified times"""

    times = []

    for i in range(runs):
        times.append(_benchmark_skew(df, column))
    
    return statistics.median(times)

In [0]:
def _benchmark_salted_skew():
    """Returns runtime of salted DF"""

    start = time.perf_counter()

    partial_count = (
        salted_df
        .groupBy("customer_id", "salted_customer_id")
        .count()
        .orderBy(F.col("count"))
    )

    final_counts = partial_count.groupBy("customer_id").agg(
        F.sum(F.col("count").alias("count")) 
    ).collect()

    end = time.perf_counter()

    return end-start

In [0]:
def _benchmark_salted_skew_multiple(runs: int = 10):
    """Executes _benchmark() specified times"""

    times = []

    for i in range(runs):
        times.append(_benchmark_salted_skew())
    
    return statistics.median(times)

In [0]:
def _run_workload(df):
    """Executes spark expression for df caching test purposes"""


    df.filter(F.col("customer_id") == "unkown").count()

    df.select("customer_id").distinct().count()

In [0]:
spark.conf.get('spark.sql.shuffle.partitions') 

In [0]:
orders_df = (
            spark.range(1_050_000)
            .withColumnRenamed("id", "order_id")
            .withColumn(
                "country",
                F.when(F.col("order_id") % 10 == 0, None)
                .when(F.col("order_id") % 3 == 0, "PL")
                .when(F.col("order_id") % 3 == 1, "DE")
                .otherwise("UK")
                )
            .withColumn(
                "amount",
                F.when(F.col("order_id") % 3 == 0, None)
                .otherwise((F.rand() * 1000).cast("double"))
            )
            )

display(orders_df)

In [0]:
(orders_df
 .withColumn("partition_id", F.spark_partition_id())
 .groupBy("partition_id")
 .count()
 .orderBy("partition_id")
 .show()
 )

In [0]:
orders_4 = orders_df.repartition(4)
print(_benchmark_multiple(orders_4))

orders_8 = orders_df.repartition(8)
print(_benchmark_multiple(orders_8))

orders_16 = orders_df.repartition(16)
print(_benchmark_multiple(orders_16))

orders_32 = orders_df.repartition(32)
print(_benchmark_multiple(orders_32))

In [0]:
# CATALYST OPTIMIZER SHUFFLE PLAN

grouped_df = (
    orders_df
    .groupBy("country")
    .count()
)

grouped_df.explain("formatted")

In [0]:
skewed_df = (
    spark.range(5_000_000)
    .withColumn("customer_id", F.when(F.col("id") % 10 != 0, F.lit("unkown")
                                      ).otherwise(
                                          F.concat(F.lit("customer_"), F.col("id"))
                                      )
                )
)
display(skewed_df)

In [0]:
display(skewed_df.groupBy("customer_id") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .limit(10))

In [0]:
result = skewed_df.groupBy("customer_id").count()

result.collect()

result.explain("formatted")

In [0]:
_benchmark_skew_multiple(skewed_df, "customer_id")

In [0]:
# SALTING

salted_df = skewed_df.withColumn("salted_customer_id", F.when(
    F.col("customer_id") == "unkown", F.concat(F.lit("unkown_"), (F.col("id") % 8).cast("string"))
    ).otherwise(F.col("customer_id"))
)
display(salted_df)

salted_df.groupBy("salted_customer_id") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(10)

          


In [0]:
# Salting decreased the runtime (- 5.9 %)

_benchmark_salted_skew_multiple()

In [0]:
# small_df
lookup_df = spark.createDataFrame([
        ("PL", "Poland"),
        ("DE", "Germany"),
        ("UK", "United Kingdom")
        ],
        ["country", "country_name"]                     
)
display(lookup_df)

In [0]:
# Catalyst optimizer automatically broadcasted this join
default_join = orders_df.join(lookup_df, on="country")
default_join.explain("formatted")

In [0]:
broadcast_join = orders_df.join(F.broadcast(lookup_df), on="country")
broadcast_join.explain("formatted")

In [0]:
# large_df
customers_df = spark.range(1_000_000).withColumnRenamed("id", "customer_id") \
    .withColumn("segment", 
                F.concat(
                    F.lit("segment_"), 
                    (F.col("customer_id") % 10).cast("string")
                    )
                )
    
orders_join_df = orders_df.withColumn(
    "customer_id", F.col("order_id") % 1_000_000
    )

display(orders_join_df)

In [0]:
# Catalyst optimizer choosed ShuffledHashJoin

start = time.perf_counter()

large_join = orders_join_df.join(
    customers_df, on="customer_id"
)


large_join.explain("formatted")

In [0]:
#  No cache
test_df = skewed_df

start_time = time.perf_counter()

_run_workload(test_df)

end_time = time.perf_counter()

print(f"No cache: {end_time - start_time}")


In [0]:
# Cache - runtime increased because cost of caching is higher than cost of creating skewed_df and operations on this df.
test_cache_df = skewed_df.cache()

test_cache_df.count()

start_time = time.perf_counter()

_run_workload(test_cache_df)

end_time = time.perf_counter()

print(f"No cache: {end_time - start_time}")

In [0]:
# spark.conf.get("spark.sql.adaptive.enabled")

# spark.conf.get("spark.sql.shuffle.partitions")

spark.conf.set("spark.sql.shuffle.partitions", 200)
# spark.conf.get("spark.sql.shuffle.partitions")

aqe_df = (
    orders_df
    .groupBy("country")
    .count()
)
aqe_df.collect()

aqe_df.explain("formatted")